# 02. Bronze Layer Ingestion

## Banking Transaction Lakehouse

**Purpose:** Ingest incoming transaction records into the Bronze Delta table while preserving the original source data.

### Scope
- Prepare incoming transaction data.
- Apply batch identification.
- Ingest records into Bronze using the reusable processor.
- Verify record counts and ingestion results.
- Demonstrate sequential retry behaviour.

**Implementation:** Bronze ingestion logic is maintained in `src/bronze/bronze_processor.py`.

In [0]:
from pyspark.sql import functions as F
from src.ingestion.sample_source import create_sample_transactions
from src.bronze.bronze_processor import ingest_to_bronze

BRONZE_TABLE = "workspace.default.dev_bronze_transactions"

batch_id = "bronze_demo_20261106_01"

print("Bronze ingestion notebook initialized.")

In [0]:
# Prepare incoming source records
incoming_df = create_sample_transactions(spark)

# Ingest into Bronze
bronze_batch_df = ingest_to_bronze(
    spark=spark,
    incoming_df=incoming_df,
    batch_id=batch_id,
    bronze_table=BRONZE_TABLE
)

# Verify ingestion
print("Bronze records:", bronze_batch_df.count())

display(bronze_batch_df)

In [0]:
# Retry Bronze ingestion using the same batch ID and input data
retry_df = ingest_to_bronze(
    spark=spark,
    incoming_df=incoming_df,
    batch_id=batch_id,
    bronze_table=BRONZE_TABLE
)

print("Records after retry:", retry_df.count())

display(retry_df)

In [0]:
# Verify records persisted for this batch
bronze_verification = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("batch_id") == batch_id)
)

display(
    bronze_verification.groupBy("batch_id")
    .agg(
        F.count("*").alias("total_records"),
        F.countDistinct("transaction_id").alias("unique_transactions")
    )
)